# Signals of Prestige in Film Discourse: A Transformer-Based Approach to Predicting Best Picture Winners

### By: Ed Hou, Si Qin Huang, Alejandro Mendez

---

Click here to [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/s-huang23/nlp-oscar-predictor/blob/model_dev/oscar_predictor.ipynb)


In [1]:
!pip install kagglehub

  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
Using cached idna-3.11-py3-none-any.whl (71 kB)
Using cached urllib3-2.6.3-py3-none-any.whl (131 kB)
Using cached certifi-2026.2.25-py3-none-any.whl (153 kB)

   ----------------------------------------  0/10 [urllib3]
   ----------------------------------------  0/10 [urllib3]
   ---- -----------------------------------  1/10 [tqdm]
   -------- -------------------------------  2/10 [pyyaml]
   ------------ ---------------------------  3/10 [protobuf]
   ------------ ---------------------------  3/10 [protobuf]
   ------------ ---------------------------  3/10 [protobuf]
   ------------ ---------------------------  3/10 [protobuf]
   -------------------- -------------------  5/10 [charset_normalizer]
   ---------------------------- -----------  7/10 [requests]
   -------------------------------- -

In [2]:
import kagglehub
import os
import glob
import json
import pandas as pd
import numpy as np
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertModel, AutoModel, AutoTokenizer
from typing import Optional

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import scipy.sparse as sp

c:\Users\siqin\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Mount drive (If manually uploaded data, mount drive)
# from google.colab import drive
# drive.mount('/content/drive')

# Clone the data repository for required datasets (if not already present)
if not os.path.exists("data"):
    os.system("git clone https://github.com/s-huang23/nlp-oscar-predictor.git . --depth=1")

## Dataset (NEEDS TO BE IMPLEMENTED)

Load iMDb data from kaggle and web-scraped reviews from Metacritic

#### Metacritic data

In [ ]:
# Load metacritic dataset
# metacritic_df = pd.read_csv("/content/drive/MyDrive/metacritic_reviews.csv") # If data is uploaded to drive
metacritic_df = pd.read_csv("data/raw/metacritic_reviews.csv")
print(metacritic_df.shape)
metacritic_df.head()

(4725, 13)


,ceremony_year,release_year,film_title,winner,metacritic_slug,metacritic_url,critic_review_page,review_date,critic_score,publication,author,quote,full_review_url
0,2012,2011,The Artist,1,the-artist,https://www.metacritic.com/movie/the-artist/,https://www.metacritic.com/movie/the-artist/cr...,2012-01-20,100,New Orleans Times-Picayune,Mike Scott,"If nothing else, this is a cinematic high-wire...",http://www.nola.com/movies/index.ssf/2012/01/t...
1,2012,2011,The Artist,1,the-artist,https://www.metacritic.com/movie/the-artist/,https://www.metacritic.com/movie/the-artist/cr...,2011-12-28,88,ReelViews,James Berardinelli,"Hazanavicius isn't just making a ""silent movie...",http://www.reelviews.net/php_review_template.p...
2,2012,2011,The Artist,1,the-artist,https://www.metacritic.com/movie/the-artist/,https://www.metacritic.com/movie/the-artist/cr...,2011-12-22,100,Arizona Republic,Bill Goodykoontz,"The Artist is such an engaging, delightful fil...",http://www.azcentral.com/thingstodo/movies/art...
3,2012,2011,The Artist,1,the-artist,https://www.metacritic.com/movie/the-artist/,https://www.metacritic.com/movie/the-artist/cr...,2011-12-22,78,Austin Chronicle,Marc Savlov,What's so intensely pleasurable about The Arti...,http://www.austinchronicle.com/calendar/film/2...
4,2012,2011,The Artist,1,the-artist,https://www.metacritic.com/movie/the-artist/,https://www.metacritic.com/movie/the-artist/cr...,2011-12-22,88,Charlotte Observer,Lawrence Toppman,"Talkies may have killed silent movies, the way...",http://events.charlotteobserver.com/reviews/sh...


#### Kaggle iMDb data

In [ ]:
# Load nominees dataset
# nominees_df = pd.read_csv("/content/drive/MyDrive/nominees.csv") # If data is uploaded to drive
nominees_df   = pd.read_csv("data/nominees.csv")
print(nominees_df.shape)
nominees_df.head()

(96, 5)


,ceremony_year,film_title,release_year,winner,metacritic_slug
0,2012,The Artist,2011,1,the-artist
1,2012,The Descendants,2011,0,the-descendants
2,2012,Extremely Loud and Incredibly Close,2011,0,extremely-loud-and-incredibly-close
3,2012,The Help,2011,0,the-help
4,2012,Hugo,2011,0,hugo


In [6]:
# Download latest version of data from kaggle
path = kagglehub.dataset_download("ebiswas/imdb-review-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'imdb-review-dataset' dataset.
Path to dataset files: /kaggle/input/imdb-review-dataset


In [7]:
# Making sure title discrepancies are fixed
title_mapping = {
    "Extremely Loud and Incredibly Close": "Extremely Loud & Incredibly Close",
    "Les Miserables": "Les MisÃ©rables",
    "Once Upon a Time ... in Hollywood" : "Once Upon a Time... In Hollywood",
    "Hell or High Water" : "Hell or High Water (II)",
    "Boyhood" : "Boyhood (I)",
    "Arrival" : "Arrival (II)",
    "Get Out" : "Get Out (I)",
    "Moonlight" : "Moonlight (I)",
    "Room" : "Room (I)",
    "Spotlight" : "Spotlight (I)",
    "The Artist" : "The Artist (I)",
    "Vice" : "Vice (I)"
}

# Mapping title fixes
nominees_df["film_title"] = nominees_df["film_title"].replace(title_mapping)
metacritic_df["film_title"] = metacritic_df["film_title"].replace(title_mapping)

# Verify the fix
#print(nominees_df[nominees_df["film_title"].isin(title_mapping.values())][["film_title", "ceremony_year"]])

                           film_title  ceremony_year
0                      The Artist (I)           2012
2   Extremely Loud & Incredibly Close           2012
13                     Les MisÃ©rables           2013
29                        Boyhood (I)           2015
41                           Room (I)           2016
42                      Spotlight (I)           2016
43                       Arrival (II)           2017
46            Hell or High Water (II)           2017
51                      Moonlight (I)           2017
55                        Get Out (I)           2018
68                           Vice (I)           2019
76   Once Upon a Time... In Hollywood           2020


In [ ]:
# print(os.listdir(path))

In [8]:
# Combine all parts of json
all_files = glob.glob(os.path.join(path, "part-*.json"))

records = []
for f in all_files:
    with open(f) as fh:
        records.extend(json.load(fh))

# Convert to dataframe
reviews_df = pd.DataFrame(records)
print(reviews_df.shape)
reviews_df.head()

(5571499, 9)


,review_id,reviewer,movie,rating,review_summary,review_date,spoiler_tag,review_detail,helpful
0,rw1133942,OriginalMovieBuff21,Kill Bill: Vol. 2 (2004),8,Good follow up that answers all the questions,24 July 2005,0,"After seeing Tarantino's Kill Bill Vol: 1, I g...","[0, 1]"
1,rw1133943,sentra14,Journey to the Unknown (1968â€“ ),None,Excellent series,24 July 2005,0,"I have the entire series on video, taped mostl...","[11, 11]"
2,rw1133946,GreenwheelFan2002,The Island (2005),9,"Not just about action, but about survival...",24 July 2005,0,Once again the critics prove themselves as mor...,"[2, 5]"
3,rw1133948,itsascreambaby,Win a Date with Tad Hamilton! (2004),3,Falls under the category: seen it a million ti...,24 July 2005,0,This IS a film that has been done too many tim...,"[2, 3]"
4,rw1133949,OriginalMovieBuff21,Saturday Night Live: The Best of Chris Farley ...,10,"Before Tommy Boy and Black Sheep, there was Sa...",24 July 2005,0,Chris Farley is one of my favorite comedians a...,"[4, 4]"


In [10]:
# Extract title and year from "Movie Title (YEAR)" format
reviews_df["release_year"] = (
    reviews_df["movie"]
    .str.extract(r'\((\d{4})')   # grab the first 4-digit year after '('
    .astype(float)
    .astype("Int64")
)

reviews_df["movie"] = (
    reviews_df["movie"]
    .str.replace(r'\s*\(\d{4}[^)]*\)', '', regex=True)  # remove anything from (YEAR...  to )
    .str.strip()
)

print(reviews_df[["movie", "release_year"]].head(10))

                                           movie  release_year
0                              Kill Bill: Vol. 2          2004
1                         Journey to the Unknown          1968
2                                     The Island          2005
3                  Win a Date with Tad Hamilton!          2004
4  Saturday Night Live: The Best of Chris Farley          2000
5                                    Outlaw Star          1998
6                                    The Aviator          2004
7      Star Wars: Episode I - The Phantom Menace          1999
8                          The Amityville Horror          2005
9                                  Flying Tigers          1942


In [11]:
# Filter and merge using both columns
filtered_reviews = reviews_df[
    reviews_df["movie"].isin(nominees_df["film_title"]) &
    reviews_df["release_year"].isin(nominees_df["release_year"])
].copy()

print(f"Total reviews matched: {len(filtered_reviews)}")
print(f"Unique films matched : {filtered_reviews['movie'].nunique()} / {len(nominees_df)}")

imdb_df = filtered_reviews.merge(
    nominees_df[["film_title", "release_year", "ceremony_year", "winner"]],
    left_on=["movie", "release_year"],
    right_on=["film_title", "release_year"],
    how="left"
)

# Drop values with NAs that don't match up
imdb_df = imdb_df.dropna(subset=["winner"])
# Fix winner and ceremony_year column formatting
imdb_df["winner"] = imdb_df["winner"].astype(int)
imdb_df["ceremony_year"] = imdb_df["ceremony_year"].astype(int)

print(imdb_df.shape)
imdb_df.head()

Total reviews matched: 84642
Unique films matched : 87 / 96
(84107, 13)


,review_id,reviewer,movie,rating,review_summary,review_date,spoiler_tag,review_detail,helpful,release_year,film_title,ceremony_year,winner
0,rw2531251,galvanekps,The Help,8,You just never know...,12 December 2011,0,I have to state up front I went into watching ...,"[0, 5]",2011,The Help,2012,0
1,rw2531254,remedy305,Hugo,8,1920's Paris comes to life,12 December 2011,0,"This is a wonderful story about a boy, Hugo, w...","[4, 6]",2011,Hugo,2012,0
2,rw2531322,gnguyen102,Midnight in Paris,9,Well-written script and outstanding acting,12 December 2011,0,I must admit that I have been a fan of Woody A...,"[0, 1]",2011,Midnight in Paris,2012,0
3,rw2531410,mmobini,Hugo,7,A huge potential for an exceptional film.,12 December 2011,0,A resplendently sweeping opening shot that pro...,"[2, 5]",2011,Hugo,2012,0
4,rw2531412,Abigail_fox,Hugo,1,Couldn't wait for it to end - so dull!,12 December 2011,0,I saw this film yesterday. I didn't see it in ...,"[88, 176]",2011,Hugo,2012,0


In [12]:
# Drop unneeded columns
imdb_df = imdb_df.drop(columns=["movie", "review_summary", "spoiler_tag", "helpful"])

print(imdb_df.columns.tolist())
print(imdb_df["winner"].value_counts())
imdb_df.head()

['review_id', 'reviewer', 'rating', 'review_date', 'review_detail', 'release_year', 'film_title', 'ceremony_year', 'winner']
winner
0    73552
1    10555
Name: count, dtype: int64


,review_id,reviewer,rating,review_date,review_detail,release_year,film_title,ceremony_year,winner
0,rw2531251,galvanekps,8,12 December 2011,I have to state up front I went into watching ...,2011,The Help,2012,0
1,rw2531254,remedy305,8,12 December 2011,"This is a wonderful story about a boy, Hugo, w...",2011,Hugo,2012,0
2,rw2531322,gnguyen102,9,12 December 2011,I must admit that I have been a fan of Woody A...,2011,Midnight in Paris,2012,0
3,rw2531410,mmobini,7,12 December 2011,A resplendently sweeping opening shot that pro...,2011,Hugo,2012,0
4,rw2531412,Abigail_fox,1,12 December 2011,I saw this film yesterday. I didn't see it in ...,2011,Hugo,2012,0


## Pre-processing

Implemented steps:
  - Filter to ceremony years 2012-2020
  - Remove all reviews after ceremony date
  - Remove short or empty reviews and duplicate review text
  - Add per-film review counts and inverse review weights for volume normalization
  - Clean review text into `clean_text`

In [ ]:
# Filter to the project scope and remove post-nomination leakage.
# Window: review_date < ceremony_date.
import re

START_YEAR = 2012
END_YEAR = 2020
MIN_TEXT_CHARS = 20

# windows_path = "/content/drive/MyDrive/oscar_windows.csv"
oscar_windows = pd.read_csv("data/oscar_windows.csv", parse_dates=["nomination_date", "ceremony_date"])

def clean_text(value):
    text = "" if pd.isna(value) else str(value)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def preprocess_reviews(df, source_name, text_col):
    working = df[
        (df["ceremony_year"] >= START_YEAR) &
        (df["ceremony_year"] <= END_YEAR)
    ].copy()

    working = working.merge(oscar_windows, on="ceremony_year", how="left")
    working["review_date_parsed"] = pd.to_datetime(working["review_date"], errors="coerce")

    in_window = (
        working["review_date_parsed"].notna()
        & (working["review_date_parsed"] < working["ceremony_date"])
    )
    working = working[in_window].copy()

    working["clean_text"] = working[text_col].map(clean_text)
    working = working[working["clean_text"].str.len() >= MIN_TEXT_CHARS].copy()

    dedupe_cols = ["ceremony_year", "film_title", "clean_text"]
    if source_name == "imdb" and "reviewer" in working.columns:
        dedupe_cols.insert(2, "reviewer")
    if source_name == "metacritic" and "publication" in working.columns:
        dedupe_cols.insert(2, "publication")
    working = working.drop_duplicates(subset=dedupe_cols).copy()

    working["film_review_count"] = working.groupby(
        ["ceremony_year", "film_title"]
    )["clean_text"].transform("size")
    working["film_review_weight"] = 1.0 / working["film_review_count"]

    print(f"{source_name}: {len(working)} rows")
    print(f"{source_name}: {working[['ceremony_year', 'film_title']].drop_duplicates().shape[0]} films")
    return working


imdb_df = preprocess_reviews(imdb_df, "imdb", "review_detail")
metacritic_df = preprocess_reviews(metacritic_df, "metacritic", "quote")

## Model

**Pipeline**: <br>
BERT  â†’ [CLS] per review  â†’ WeightedAggregator â†’ review film vector <br>
BERTweet â†’ [CLS] per tweet â†’ WeightedAggregator â†’ tweet film vector <br>
Concatenate â†’ Classification Head â†’ P(win)

**Weighted Aggregator**: aggregates review embeddings from BERT and BERTweet into a single film vector

### BERT (On reviews + tweets)

In [ ]:
class OscarPredictorBERT(nn.Module):
    """
    Baseline model: single shared BERT encoder for both reviews and tweets.
    Uses mean pooling instead of learned aggregation.
    """
    def __init__(
        self,
        model_name="bert-base-uncased",
        hidden_dim=768,
        dropout=0.3
    ):
        super(OscarPredictorBERT, self).__init__()

        # Single shared encoder for both reviews and tweets
        self.encoder = AutoModel.from_pretrained(model_name)

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

    def encode(self, input_ids, attention_mask):
        """
        Encode a batch of texts and return mean-pooled CLS representations.

        input_ids     : (batch_size, num_texts, seq_len)
        attention_mask: (batch_size, num_texts, seq_len)
        returns       : (batch_size, hidden_dim)
        """
        batch_size, num_texts, seq_len = input_ids.shape

        input_ids = input_ids.reshape(-1, seq_len)
        attention_mask = attention_mask.reshape(-1, seq_len)

        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls = outputs.last_hidden_state[:, 0, :]
        cls = cls.view(batch_size, num_texts, -1)
        return cls.mean(dim=1)

    def forward(
        self,
        review_input_ids,
        review_attention_mask,
        tweet_input_ids,
        tweet_attention_mask
    ):
        review_agg = self.encode(review_input_ids, review_attention_mask)
        tweet_agg  = self.encode(tweet_input_ids,  tweet_attention_mask)

        film_vector = torch.cat([review_agg, tweet_agg], dim=1)

        return self.classifier(film_vector)

### TF-IDF + Logistic Regression

In [ ]:
class OscarPredictorTFIDF:
    """
    Baseline model: TF-IDF + Logistic Regression.
    Reviews and tweets are each vectorized separately then concatenated.
    """

    def __init__(self, max_features=10000, C=1.0):
        # Separate vectorizers so each modality gets its own vocab
        self.review_vectorizer = TfidfVectorizer(
            max_features=max_features,
            ngram_range=(1, 2),    # unigrams + bigrams
            sublinear_tf=True,     # apply log normalization to term freq
            strip_accents="unicode",
            min_df=2,              # ignore terms appearing in fewer than 2 films
        )
        self.tweet_vectorizer = TfidfVectorizer(
            max_features=max_features,
            ngram_range=(1, 2),
            sublinear_tf=True,
            strip_accents="unicode",
            min_df=2,
        )
        self.classifier = LogisticRegression(
            C=C,                   # inverse regularization strength
            max_iter=1000,
            class_weight="balanced",  # handles 1 winner vs many nominees imbalance
        )

    def _prepare_features(self, reviews, tweets, fit=False):
        """
        Vectorize reviews and tweets and concatenate into one feature matrix.

        reviews : list of str â€” one string per film (all reviews concatenated)
        tweets  : list of str â€” one string per film (all tweets concatenated)
        returns : sparse matrix (num_films, 2 * max_features)
        """
        if fit:
            review_features = self.review_vectorizer.fit_transform(reviews)
            tweet_features  = self.tweet_vectorizer.fit_transform(tweets)
        else:
            review_features = self.review_vectorizer.transform(reviews)
            tweet_features  = self.tweet_vectorizer.transform(tweets)

        # Horizontally stack review and tweet features
        return sp.hstack([review_features, tweet_features])

    def fit(self, reviews, tweets, labels):
        """
        Train the model.

        reviews : list of str, one per film
        tweets  : list of str, one per film
        labels  : list of int, 1 = winner, 0 = nominee
        """
        X = self._prepare_features(reviews, tweets, fit=True)
        self.classifier.fit(X, labels)

    def predict_proba(self, reviews, tweets):
        """
        Returns P(win) for each film.

        returns: np.array of shape (num_films,)
        """
        X = self._prepare_features(reviews, tweets, fit=False)
        return self.classifier.predict_proba(X)[:, 1]

    def predict_winner(self, reviews, tweets):
        """
        Returns the index of the predicted winner within the provided films
        (intended to be called with one year's nominees at a time).
        """
        probs = self.predict_proba(reviews, tweets)
        return np.argmax(probs)

## Evaluation (NEEDS TO BE IMPLEMENTED)
Methods:
  - Accuracy
  - Top-1, Top-5
  - Others?